In [0]:
# Create business-ready features from the Customer 360 table. 
# Import Libraries 
from pyspark.sql.functions import *

In [0]:
# Load Customer 360 
gold = spark.table("silver_customer_360")

In [0]:
# Verify Data 
display(gold.limit(10))

gold.printSchema()


CustomerID,JoinDate,Segment,Region,Age,SubscriptionType,TenureDays,AgeGroup,BillingMonth,MonthlyRevenue,TransactionCount,TotalLogins,LifetimeMinutes,AvgSession,LastLoginDate,FirstLoginDate,ActiveDays,LoginFrequency,UsageLevel,SessionCategory,RecentlyActive,TotalTickets,LastTicketDate,OpenTickets,ClosedTickets,ComplaintScore,SupportLevel,OpenTicketRatio,SupportRisk,Churn,ChurnStatus,RetentionFlag
100030,2022-06-19,PREMIUM,INDIA,48,MONTHLY,1468,45-59,2024-06,2835.29,1,6,447,74.5,2025-07-16,2024-04-20,452,0.01,LOW,LONG,0,3,2024-07-22,0,3,6,LOW,0.0,LOW,0,ACTIVE,1
100036,2024-09-12,BASIC,USA,43,MONTHLY,652,35-44,2024-01,2912.39,1,7,682,97.42857142857143,2025-05-05,2024-03-16,415,0.02,LOW,LONG,0,3,2025-12-03,1,2,11,LOW,0.33,LOW,0,ACTIVE,1
100042,2023-11-02,BASIC,UK,55,MONTHLY,967,45-59,2024-11,3319.43,1,8,865,108.125,2025-09-08,2024-02-16,570,0.01,LOW,LONG,0,3,2024-05-21,0,3,6,LOW,0.0,LOW,0,ACTIVE,1
100043,2021-12-10,BASIC,USA,35,MONTHLY,1659,35-44,2024-11,2278.93,1,4,290,72.5,2025-04-07,2024-03-26,377,0.01,LOW,LONG,0,2,2025-05-30,0,2,4,LOW,0.0,LOW,0,ACTIVE,1
100060,2025-01-19,BASIC,CANADA,53,MONTHLY,523,45-59,2025-04,5080.76,1,5,652,130.4,2025-12-03,2024-03-16,627,0.01,LOW,LONG,0,1,2025-08-04,0,1,2,LOW,0.0,LOW,0,ACTIVE,1
100068,2020-06-01,BASIC,UK,19,MONTHLY,2216,18-24,2024-08,1825.58,1,10,762,76.2,2025-12-03,2024-02-14,658,0.02,LOW,LONG,0,2,2025-05-30,0,2,4,LOW,0.0,LOW,0,ACTIVE,1
100082,2020-11-17,BASIC,CANADA,47,MONTHLY,2047,45-59,2025-10,5793.53,1,3,433,144.33333333333334,2025-12-17,2025-01-15,336,0.01,LOW,LONG,0,0,null,0,0,0,LOW,null,LOW,0,ACTIVE,1
100084,2023-06-13,BASIC,UK,61,MONTHLY,1109,60+,2024-03,1674.21,1,3,281,93.66666666666667,2025-03-21,2024-03-06,380,0.01,LOW,LONG,0,2,2024-08-05,1,1,9,LOW,0.5,LOW,0,ACTIVE,1
100135,2021-08-30,BASIC,CANADA,60,MONTHLY,1761,60+,2025-06,1854.49,1,5,247,49.4,2025-11-07,2024-07-14,481,0.01,LOW,MEDIUM,0,4,2025-08-24,1,3,13,LOW,0.25,LOW,0,ACTIVE,1
100140,2025-05-19,BASIC,CANADA,18,MONTHLY,403,18-24,2024-12,1880.07,1,5,363,72.6,2025-11-28,2024-08-28,457,0.01,LOW,LONG,0,2,2024-04-02,1,1,9,LOW,0.5,LOW,1,CHURNED,0


root
 |-- CustomerID: long (nullable = true)
 |-- JoinDate: date (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- SubscriptionType: string (nullable = true)
 |-- TenureDays: integer (nullable = true)
 |-- AgeGroup: string (nullable = true)
 |-- BillingMonth: string (nullable = true)
 |-- MonthlyRevenue: double (nullable = true)
 |-- TransactionCount: long (nullable = true)
 |-- TotalLogins: long (nullable = true)
 |-- LifetimeMinutes: long (nullable = true)
 |-- AvgSession: double (nullable = true)
 |-- LastLoginDate: date (nullable = true)
 |-- FirstLoginDate: date (nullable = true)
 |-- ActiveDays: integer (nullable = true)
 |-- LoginFrequency: double (nullable = true)
 |-- UsageLevel: string (nullable = true)
 |-- SessionCategory: string (nullable = true)
 |-- RecentlyActive: integer (nullable = true)
 |-- TotalTickets: long (nullable = true)
 |-- LastTicketDate: date (nullable = true)
 |-- OpenTicke

In [0]:
# Revenue Per Login 
gold = gold.withColumn(
    "RevenuePerLogin",
    round(
        col("MonthlyRevenue") /
        when(col("TotalLogins")==0,1)
        .otherwise(col("TotalLogins")),
        2
    )
)

In [0]:
# Tickets Per Login (complaints occur per login) 
gold = gold.withColumn(
    "TicketsPerLogin",
    round(
        col("TotalTickets") /
        when(col("TotalLogins")==0,1)
        .otherwise(col("TotalLogins")),
        2
    )
)

In [0]:
# Customer Engagement Score 
gold = gold.withColumn(

    "EngagementScore",

    round(

        col("TotalLogins")*0.40 +

        col("LifetimeMinutes")*0.30 +

        col("MonthlyRevenue")*0.30,

        2
    )
)

In [0]:
# Complaint Ratio
# How many complaints per ticket 
gold = gold.withColumn(

    "ComplaintRatio",

    round(

        col("ComplaintScore")/

        when(col("TotalTickets")==0,1)

        .otherwise(col("TotalTickets")),

        2
    )
)

In [0]:
# Revenue Category 
gold = gold.withColumn(

    "RevenueCategory",

    when(col("MonthlyRevenue")>=10000,"PLATINUM")

    .when(col("MonthlyRevenue")>=5000,"GOLD")

    .when(col("MonthlyRevenue")>=2000,"SILVER")

    .otherwise("BRONZE")
)

In [0]:
# Customer Lifetime Segment 
gold = gold.withColumn(

    "LifetimeSegment",

    when(col("TenureDays")>=730,"LOYAL")

    .when(col("TenureDays")>=365,"ESTABLISHED")

    .otherwise("NEW")
)

In [0]:
# Login Category 
gold = gold.withColumn(

    "LoginCategory",

    when(col("TotalLogins")>=200,"VERY ACTIVE")

    .when(col("TotalLogins")>=100,"ACTIVE")

    .when(col("TotalLogins")>=50,"MODERATE")

    .otherwise("LOW")
)

In [0]:
# Usage Category 
gold = gold.withColumn(

    "UsageCategory",

    when(col("LifetimeMinutes")>=10000,"HEAVY")

    .when(col("LifetimeMinutes")>=5000,"MEDIUM")

    .otherwise("LIGHT")
)

In [0]:
# Churn Risk Score 
gold = gold.withColumn(

    "RiskScore",

    round(

        (col("ComplaintScore")*0.5)

        +

        ((100-col("TotalLogins"))*0.3)

        +

        ((5000-col("MonthlyRevenue"))/1000),

        2
    )
)

In [0]:
# High Risk Flag 
gold = gold.withColumn(

    "HighRiskCustomer",

    when(col("RiskScore")>=50,1)

    .otherwise(0)
)

In [0]:
# Customer Health Score 
gold = gold.withColumn(

    "CustomerHealth",

    round(

        col("EngagementScore")

        -

        col("ComplaintScore"),

        2
    )
)

In [0]:
# Health Category 
gold = gold.withColumn(

    "HealthCategory",

    when(col("CustomerHealth")>=100,"EXCELLENT")

    .when(col("CustomerHealth")>=60,"GOOD")

    .when(col("CustomerHealth")>=30,"AVERAGE")

    .otherwise("POOR")
)

In [0]:
# Loyalty Score

gold = gold.withColumn(

    "LoyaltyScore",

    round(

        col("TenureDays")*0.2 +

        col("MonthlyRevenue")*0.3 +

        col("TotalLogins")*0.5,

        2
    )
)

In [0]:
# Validate Output 
display(gold.limit(20))

CustomerID,JoinDate,Segment,Region,Age,SubscriptionType,TenureDays,AgeGroup,BillingMonth,MonthlyRevenue,TransactionCount,TotalLogins,LifetimeMinutes,AvgSession,LastLoginDate,FirstLoginDate,ActiveDays,LoginFrequency,UsageLevel,SessionCategory,RecentlyActive,TotalTickets,LastTicketDate,OpenTickets,ClosedTickets,ComplaintScore,SupportLevel,OpenTicketRatio,SupportRisk,Churn,ChurnStatus,RetentionFlag,RevenuePerLogin,TicketsPerLogin,EngagementScore,ComplaintRatio,RevenueCategory,LifetimeSegment,LoginCategory,UsageCategory,RiskScore,HighRiskCustomer,CustomerHealth,HealthCategory,LoyaltyScore
100030,2022-06-19,PREMIUM,INDIA,48,MONTHLY,1468,45-59,2024-06,2835.29,1,6,447,74.5,2025-07-16,2024-04-20,452,0.01,LOW,LONG,0,3,2024-07-22,0,3,6,LOW,0.0,LOW,0,ACTIVE,1,472.55,0.5,987.09,2.0,SILVER,LOYAL,LOW,LIGHT,33.36,0,981.09,EXCELLENT,1147.19
100036,2024-09-12,BASIC,USA,43,MONTHLY,652,35-44,2024-01,2912.39,1,7,682,97.42857142857143,2025-05-05,2024-03-16,415,0.02,LOW,LONG,0,3,2025-12-03,1,2,11,LOW,0.33,LOW,0,ACTIVE,1,416.06,0.43,1081.12,3.67,SILVER,ESTABLISHED,LOW,LIGHT,35.49,0,1070.12,EXCELLENT,1007.62
100042,2023-11-02,BASIC,UK,55,MONTHLY,967,45-59,2024-11,3319.43,1,8,865,108.125,2025-09-08,2024-02-16,570,0.01,LOW,LONG,0,3,2024-05-21,0,3,6,LOW,0.0,LOW,0,ACTIVE,1,414.93,0.38,1258.53,2.0,SILVER,LOYAL,LOW,LIGHT,32.28,0,1252.53,EXCELLENT,1193.23
100043,2021-12-10,BASIC,USA,35,MONTHLY,1659,35-44,2024-11,2278.93,1,4,290,72.5,2025-04-07,2024-03-26,377,0.01,LOW,LONG,0,2,2025-05-30,0,2,4,LOW,0.0,LOW,0,ACTIVE,1,569.73,0.5,772.28,2.0,SILVER,LOYAL,LOW,LIGHT,33.52,0,768.28,EXCELLENT,1017.48
100060,2025-01-19,BASIC,CANADA,53,MONTHLY,523,45-59,2025-04,5080.76,1,5,652,130.4,2025-12-03,2024-03-16,627,0.01,LOW,LONG,0,1,2025-08-04,0,1,2,LOW,0.0,LOW,0,ACTIVE,1,1016.15,0.2,1721.83,2.0,GOLD,ESTABLISHED,LOW,LIGHT,29.42,0,1719.83,EXCELLENT,1631.33
100068,2020-06-01,BASIC,UK,19,MONTHLY,2216,18-24,2024-08,1825.58,1,10,762,76.2,2025-12-03,2024-02-14,658,0.02,LOW,LONG,0,2,2025-05-30,0,2,4,LOW,0.0,LOW,0,ACTIVE,1,182.56,0.2,780.27,2.0,BRONZE,LOYAL,LOW,LIGHT,32.17,0,776.27,EXCELLENT,995.87
100082,2020-11-17,BASIC,CANADA,47,MONTHLY,2047,45-59,2025-10,5793.53,1,3,433,144.33333333333334,2025-12-17,2025-01-15,336,0.01,LOW,LONG,0,0,null,0,0,0,LOW,null,LOW,0,ACTIVE,1,1931.18,0.0,1869.16,0.0,GOLD,LOYAL,LOW,LIGHT,28.31,0,1869.16,EXCELLENT,2148.96
100084,2023-06-13,BASIC,UK,61,MONTHLY,1109,60+,2024-03,1674.21,1,3,281,93.66666666666667,2025-03-21,2024-03-06,380,0.01,LOW,LONG,0,2,2024-08-05,1,1,9,LOW,0.5,LOW,0,ACTIVE,1,558.07,0.67,587.76,4.5,BRONZE,LOYAL,LOW,LIGHT,36.93,0,578.76,EXCELLENT,725.56
100135,2021-08-30,BASIC,CANADA,60,MONTHLY,1761,60+,2025-06,1854.49,1,5,247,49.4,2025-11-07,2024-07-14,481,0.01,LOW,MEDIUM,0,4,2025-08-24,1,3,13,LOW,0.25,LOW,0,ACTIVE,1,370.9,0.8,632.45,3.25,BRONZE,LOYAL,LOW,LIGHT,38.15,0,619.45,EXCELLENT,911.05
100140,2025-05-19,BASIC,CANADA,18,MONTHLY,403,18-24,2024-12,1880.07,1,5,363,72.6,2025-11-28,2024-08-28,457,0.01,LOW,LONG,0,2,2024-04-02,1,1,9,LOW,0.5,LOW,1,CHURNED,0,376.01,0.4,674.92,4.5,BRONZE,ESTABLISHED,LOW,LIGHT,36.12,0,665.92,EXCELLENT,647.12


In [0]:
# Verify Schema 
gold.printSchema()

root
 |-- CustomerID: long (nullable = true)
 |-- JoinDate: date (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- SubscriptionType: string (nullable = true)
 |-- TenureDays: integer (nullable = true)
 |-- AgeGroup: string (nullable = true)
 |-- BillingMonth: string (nullable = true)
 |-- MonthlyRevenue: double (nullable = true)
 |-- TransactionCount: long (nullable = true)
 |-- TotalLogins: long (nullable = true)
 |-- LifetimeMinutes: long (nullable = true)
 |-- AvgSession: double (nullable = true)
 |-- LastLoginDate: date (nullable = true)
 |-- FirstLoginDate: date (nullable = true)
 |-- ActiveDays: integer (nullable = true)
 |-- LoginFrequency: double (nullable = true)
 |-- UsageLevel: string (nullable = true)
 |-- SessionCategory: string (nullable = true)
 |-- RecentlyActive: integer (nullable = true)
 |-- TotalTickets: long (nullable = true)
 |-- LastTicketDate: date (nullable = true)
 |-- OpenTicke

In [0]:
# Basic Statistics 
gold.describe().show()

+-------+------------------+-------+------+------------------+----------------+-----------------+--------+------------+-----------------+------------------+------------------+------------------+------------------+------------------+--------------------+----------+---------------+--------------+------------------+-------------------+------------------+-----------------+------------+-------------------+-----------+------------------+-----------+------------------+-----------------+------------------+------------------+------------------+---------------+---------------+-------------+-------------+------------------+--------------------+------------------+--------------+------------------+
|summary|        CustomerID|Segment|Region|               Age|SubscriptionType|       TenureDays|AgeGroup|BillingMonth|   MonthlyRevenue|  TransactionCount|       TotalLogins|   LifetimeMinutes|        AvgSession|        ActiveDays|      LoginFrequency|UsageLevel|SessionCategory|RecentlyActive|      Tota

In [0]:
# Save Gold Layer 
gold.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("gold_retention_features")

In [0]:
# Validate Saved Table  
display(
    spark.table("gold_retention_features")
)


CustomerID,JoinDate,Segment,Region,Age,SubscriptionType,TenureDays,AgeGroup,BillingMonth,MonthlyRevenue,TransactionCount,TotalLogins,LifetimeMinutes,AvgSession,LastLoginDate,FirstLoginDate,ActiveDays,LoginFrequency,UsageLevel,SessionCategory,RecentlyActive,TotalTickets,LastTicketDate,OpenTickets,ClosedTickets,ComplaintScore,SupportLevel,OpenTicketRatio,SupportRisk,Churn,ChurnStatus,RetentionFlag,RevenuePerLogin,TicketsPerLogin,EngagementScore,ComplaintRatio,RevenueCategory,LifetimeSegment,LoginCategory,UsageCategory,RiskScore,HighRiskCustomer,CustomerHealth,HealthCategory,LoyaltyScore
100030,2022-06-19,PREMIUM,INDIA,48,MONTHLY,1468,45-59,2024-06,2835.29,1,6,447,74.5,2025-07-16,2024-04-20,452,0.01,LOW,LONG,0,3,2024-07-22,0,3,6,LOW,0.0,LOW,0,ACTIVE,1,472.55,0.5,987.09,2.0,SILVER,LOYAL,LOW,LIGHT,33.36,0,981.09,EXCELLENT,1147.19
100036,2024-09-12,BASIC,USA,43,MONTHLY,652,35-44,2024-01,2912.39,1,7,682,97.42857142857143,2025-05-05,2024-03-16,415,0.02,LOW,LONG,0,3,2025-12-03,1,2,11,LOW,0.33,LOW,0,ACTIVE,1,416.06,0.43,1081.12,3.67,SILVER,ESTABLISHED,LOW,LIGHT,35.49,0,1070.12,EXCELLENT,1007.62
100042,2023-11-02,BASIC,UK,55,MONTHLY,967,45-59,2024-11,3319.43,1,8,865,108.125,2025-09-08,2024-02-16,570,0.01,LOW,LONG,0,3,2024-05-21,0,3,6,LOW,0.0,LOW,0,ACTIVE,1,414.93,0.38,1258.53,2.0,SILVER,LOYAL,LOW,LIGHT,32.28,0,1252.53,EXCELLENT,1193.23
100043,2021-12-10,BASIC,USA,35,MONTHLY,1659,35-44,2024-11,2278.93,1,4,290,72.5,2025-04-07,2024-03-26,377,0.01,LOW,LONG,0,2,2025-05-30,0,2,4,LOW,0.0,LOW,0,ACTIVE,1,569.73,0.5,772.28,2.0,SILVER,LOYAL,LOW,LIGHT,33.52,0,768.28,EXCELLENT,1017.48
100060,2025-01-19,BASIC,CANADA,53,MONTHLY,523,45-59,2025-04,5080.76,1,5,652,130.4,2025-12-03,2024-03-16,627,0.01,LOW,LONG,0,1,2025-08-04,0,1,2,LOW,0.0,LOW,0,ACTIVE,1,1016.15,0.2,1721.83,2.0,GOLD,ESTABLISHED,LOW,LIGHT,29.42,0,1719.83,EXCELLENT,1631.33
100068,2020-06-01,BASIC,UK,19,MONTHLY,2216,18-24,2024-08,1825.58,1,10,762,76.2,2025-12-03,2024-02-14,658,0.02,LOW,LONG,0,2,2025-05-30,0,2,4,LOW,0.0,LOW,0,ACTIVE,1,182.56,0.2,780.27,2.0,BRONZE,LOYAL,LOW,LIGHT,32.17,0,776.27,EXCELLENT,995.87
100082,2020-11-17,BASIC,CANADA,47,MONTHLY,2047,45-59,2025-10,5793.53,1,3,433,144.33333333333334,2025-12-17,2025-01-15,336,0.01,LOW,LONG,0,0,null,0,0,0,LOW,null,LOW,0,ACTIVE,1,1931.18,0.0,1869.16,0.0,GOLD,LOYAL,LOW,LIGHT,28.31,0,1869.16,EXCELLENT,2148.96
100084,2023-06-13,BASIC,UK,61,MONTHLY,1109,60+,2024-03,1674.21,1,3,281,93.66666666666667,2025-03-21,2024-03-06,380,0.01,LOW,LONG,0,2,2024-08-05,1,1,9,LOW,0.5,LOW,0,ACTIVE,1,558.07,0.67,587.76,4.5,BRONZE,LOYAL,LOW,LIGHT,36.93,0,578.76,EXCELLENT,725.56
100135,2021-08-30,BASIC,CANADA,60,MONTHLY,1761,60+,2025-06,1854.49,1,5,247,49.4,2025-11-07,2024-07-14,481,0.01,LOW,MEDIUM,0,4,2025-08-24,1,3,13,LOW,0.25,LOW,0,ACTIVE,1,370.9,0.8,632.45,3.25,BRONZE,LOYAL,LOW,LIGHT,38.15,0,619.45,EXCELLENT,911.05
100140,2025-05-19,BASIC,CANADA,18,MONTHLY,403,18-24,2024-12,1880.07,1,5,363,72.6,2025-11-28,2024-08-28,457,0.01,LOW,LONG,0,2,2024-04-02,1,1,9,LOW,0.5,LOW,1,CHURNED,0,376.01,0.4,674.92,4.5,BRONZE,ESTABLISHED,LOW,LIGHT,36.12,0,665.92,EXCELLENT,647.12


In [0]:
#